# 放弃使用可执行文件进行操作，使用脚本对ppac进行刻度

In [1]:
%jsroot on
TCanvas *c1 = new TCanvas();
gPad->SetGrid();

In [ ]:
//读取，输入的root文件

int run_number = 510;


TString InputPath, OutputPath, infile, outfile;  
InputPath = "/data/d1/wanww/data/T0_hit/";
OutputPath = "/data/d1/wanww/data/ppac_correlation/";
infile.Form("%sT0-ppac-hit-%04d.root", InputPath.Data(), run_number);
outfile.Form("%sppac-correlation-%04d.root", OutputPath.Data(), run_number);
cout << infile << endl;
cout << outfile << endl;

TFile *ipf = new TFile(infile);
if(!ipf->IsOpen()) 
{
    cout<<"Cannot open input file: "<<infile<<endl;
    return -1;
}
TTree *tree = (TTree*)ipf->Get("tree");

//output
TFile *opf = new TFile(outfile,"RECREATE");
TTree *opt = new TTree("tree","ppac_figure");


//设置Branch
TCut p1_ca = "fp_fp1_a[0]>0";
TCut p1_cx1 = "fp_fp1_x1[0]>0" && p1_ca;
TCut p1_cx2 = "fp_fp1_x2[0]>0" && p1_ca;
TCut p1_cy1 = "fp_fp1_y1[0]>0" && p1_ca;
TCut p1_cy2 = "fp_fp1_y2[0]>0" && p1_ca;

TCut p2_ca = "fp_fp2_a[0]>0";
TCut p2_cx1 = "fp_fp2_x1[0]>0" && p2_ca;
TCut p2_cx2 = "fp_fp2_x2[0]>0" && p2_ca;
TCut p2_cy1 = "fp_fp2_y1[0]>0" && p2_ca;
TCut p2_cy2 = "fp_fp2_y2[0]>0" && p2_ca;

TCut p3_ca = "fp_fp3_a[0]>0";
TCut p3_cx1 = "fp_fp3_x1[0]>0" && p3_ca;
TCut p3_cx2 = "fp_fp3_x2[0]>0" && p3_ca;
TCut p3_cy1 = "fp_fp3_y1[0]>0" && p3_ca;
TCut p3_cy2 = "fp_fp3_y2[0]>0" && p3_ca;

// PPAC1
tree->SetAlias("p1_Tx1", "fp_fp1_x1[0]");
tree->SetAlias("p1_Tx2", "fp_fp1_x2[0]");
tree->SetAlias("p1_Ty1", "fp_fp1_y1[0]");
tree->SetAlias("p1_Ty2", "fp_fp1_y2[0]");
tree->SetAlias("p1_Ta", "fp_fp1_a[0]");

tree->SetAlias("p1_Tx_Delay", "p1_Tx1+p1_Tx2 - 2*p1_Ta");
tree->SetAlias("p1_Ty_Delay", "p1_Ty1+p1_Ty2 - 2*p1_Ta");
tree->SetAlias("p1_dtx12", "p1_Tx1 - p1_Tx2");
tree->SetAlias("p1_dty21", "p1_Ty2 - p1_Ty1");

//PPAC2
tree->SetAlias("p2_Tx1", "fp_fp2_x1[0]");
tree->SetAlias("p2_Tx2", "fp_fp2_x2[0]");
tree->SetAlias("p2_Ty1", "fp_fp2_y1[0]");
tree->SetAlias("p2_Ty2", "fp_fp2_y2[0]");
tree->SetAlias("p2_Ta", "fp_fp2_a[0]");

tree->SetAlias("p2_Tx_Delay", "p2_Tx1+p2_Tx2 - 2*p2_Ta");
tree->SetAlias("p2_Ty_Delay", "p2_Ty1+p2_Ty2 - 2*p2_Ta");
tree->SetAlias("p2_dtx12", "p2_Tx1 - p2_Tx2");
tree->SetAlias("p2_dty21", "p2_Ty2 - p2_Ty1");

//PPAC3
tree->SetAlias("p3_Tx1", "fp_fp3_x1[0]");
tree->SetAlias("p3_Tx2", "fp_fp3_x2[0]");
tree->SetAlias("p3_Ty1", "fp_fp3_y1[0]");
tree->SetAlias("p3_Ty2", "fp_fp3_y2[0]");
tree->SetAlias("p3_Ta", "fp_fp3_a[0]");

tree->SetAlias("p3_Tx_Delay", "p3_Tx1+p3_Tx2 - 2*p3_Ta");
tree->SetAlias("p3_Ty_Delay", "p3_Ty1+p3_Ty2 - 2*p3_Ta");
tree->SetAlias("p3_dtx12", "p3_Tx1 - p3_Tx2");
tree->SetAlias("p3_dty21", "p3_Ty2 - p3_Ty1");



In [ ]:
// 定义输出数据
double ppac_xpeak[3], ppac_ypeak[3];     //延迟时间的峰位
double ppac_xsigma[3], ppac_ysigma[3];   //延迟时间的峰宽
double position[8] = {-4, -3, -2, -1, 0, 1, 2, 3};

double ppac_dtx[3][8], ppac_dty[3][8];  //时间差谱的峰位
// double ppac_dtx_count[3][8], ppac_dty_count[3][8];  //时间差谱峰位处的计数

double ppac_corex[3][3], ppac_corey[3][3];   //拟合系数， p0, p1, chi2/ndf

TH1D *ppac1_Tx_Delay;
TH1D *ppac1_Ty_Delay;
TH1D *ppac1_dtx12;
TH1D *ppac1_dty21;

TH1D *ppac2_Tx_Delay;
TH1D *ppac2_Ty_Delay;
TH1D *ppac2_dtx12;
TH1D *ppac2_dty21;

TH1D *ppac3_Tx_Delay;
TH1D *ppac3_Ty_Delay;
TH1D *ppac3_dtx12;
TH1D *ppac3_dty21;

//定义中间数据
TSpectrum *s_ppac1_Tx_Delay = new TSpectrum(5);
TSpectrum *s_ppac1_Ty_Delay = new TSpectrum(5);
TSpectrum *s_ppac1_dtx12    = new TSpectrum(50);
TSpectrum *s_ppac1_dty21    = new TSpectrum(50);

TSpectrum *s_ppac2_Tx_Delay = new TSpectrum(5);
TSpectrum *s_ppac2_Ty_Delay = new TSpectrum(5);
TSpectrum *s_ppac2_dtx12    = new TSpectrum(50);
TSpectrum *s_ppac2_dty21    = new TSpectrum(50);

TSpectrum *s_ppac3_Tx_Delay = new TSpectrum(5);
TSpectrum *s_ppac3_Ty_Delay = new TSpectrum(5);
TSpectrum *s_ppac3_dtx12    = new TSpectrum(50);
TSpectrum *s_ppac3_dty21    = new TSpectrum(50);

TF1 *fx_delay[3];
TF1 *fy_delay[3];

TF1 *fx_dt[3][8];
TF1 *fy_dt[3][8];

TGraph *gr;
TF1 *fit_x[3];
TF1 *fit_y[3];

double *dt;
// double *dt_count;
int nfound;
TString spileup;
TCut cpileup;
vector<int> v1;



In [ ]:
void select(double num1=3.25, double num2=3.75)
{
    for(int i=2; i<nfound-1; i++)
    {
        float dp0 = dt[i-1] - dt[i-2];
        float dp1 = dt[i] - dt[i-1];
        float dp2 = dt[i+1] - dt[i];
    
        bool j2 = dp1<num1 && dp0>num2 && dp2>num2;
        if (j2)
        {
            cout << i << endl;
            v1.push_back(i);
        }
    }
}

## ppac1_Tx_Delay

In [ ]:
//ppac1_Tx_Delay
tree->Draw("p1_Tx_Delay>>ppac1_Tx_Delay(6000,-200,400)", p1_cx1 && p1_cx2, "goff");
ppac1_Tx_Delay = (TH1D*) gROOT->FindObject("ppac1_Tx_Delay");

nfound = s_ppac1_Tx_Delay->Search(ppac1_Tx_Delay,0.7,"",0.9);
dt = s_ppac1_Tx_Delay->GetPositionX();
ppac1_Tx_Delay->Fit("gaus","Q","", dt[0]-1.2, dt[0]+1.2);
fx_delay[0] = ppac1_Tx_Delay->GetFunction("gaus");
ppac_xpeak[0] = fx_delay[0]->GetParameter(1);
ppac_xsigma[0] = fx_delay[0]->GetParameter(2);

cout << ppac_xpeak[0] << " + " << ppac_xsigma[0] << endl;
gPad->SetLogy();
ppac1_Tx_Delay->Draw();
c1->Draw();

## ppac1_dtx12

In [ ]:
spileup.Form("abs(p1_Tx_Delay-%f)<3*%f",ppac_xpeak[0],ppac_xsigma[0]);
TCut cpileup = spileup.Data();
tree->Draw("p1_dtx12>>ppac1_dtx12(4000,-200,200)", p1_cx1 && p1_cx2 && cpileup, "goff");
ppac1_dtx12= (TH1D*) gROOT->FindObject("ppac1_dtx12");

In [ ]:
nfound = s_ppac1_dtx12->Search(ppac1_dtx12,2,"",0.2); //寻峰阈值
dt = s_ppac1_dtx12->GetPositionX();
// dx_count = s_dtx12->GetPositionY();
sort(dt,dt+nfound);
select(3.25,3.75);
for (int i=0; i<8; i++)
{
//     cout << dt[i+v1[0]] << endl;
    fx_dt[0][i] = new TF1(Form("f0%d", i), "gaus");
    ppac1_dtx12->Fit(fx_dt[0][i],"SQ+","",dt[i+v1[0]+1]-0.6,dt[i+v1[0]+1]+0.6);//fp1
//     fx[count] = dtx12->GetFunction("gaus");
    ppac_dtx[0][i] = fx_dt[0][i]->GetParameter(1);
    // xsigma[i] = fx[i]->GetParameter(2);
    // cout << xpeak[i] <<" + "<< xsigma << endl;
}

gPad->SetLogy(0);
ppac1_dtx12->Draw();
c1->Draw();
v1.clear();


In [ ]:
gr = new TGraph(8,ppac_dtx[0],position);
gr->Draw("AQ*");
gr->Fit("pol1","Q");
fit_x[0] = gr->GetFunction("pol1");

ppac_corex[0][0] = fit_x[0]->GetParameter(0);
ppac_corex[0][1] = fit_x[0]->GetParameter(1);
ppac_corex[0][2] = fit_x[0]->GetChisquare()/fit_x[0]->GetNDF();
cout << Form("p0 = %.6f, p1 = %.6f, chi2/ndf = %.6f", ppac_corex[0][0], ppac_corex[0][1], ppac_corex[0][2]) << endl;

c1->Draw();

## ppac1_Ty_Delay

In [ ]:
tree->Draw("p1_Ty_Delay>>ppac1_Ty_Delay(6000,-200,400)", p1_cy1 && p1_cy2, "goff");
ppac1_Ty_Delay = (TH1D*) gROOT->FindObject("ppac1_Ty_Delay");

In [ ]:
nfound = s_ppac1_Ty_Delay->Search(ppac1_Ty_Delay,0.7,"",0.9);
dt = s_ppac1_Ty_Delay->GetPositionX();
ppac1_Ty_Delay->Fit("gaus","Q","", dt[0]-1.2, dt[0]+1.2);
fy_delay[0] = ppac1_Ty_Delay->GetFunction("gaus");
ppac_ypeak[0] = fy_delay[0]->GetParameter(1);
ppac_ysigma[0] = fy_delay[0]->GetParameter(2);

cout << ppac_ypeak[0] << " + " << ppac_ysigma[0] << endl;
gPad->SetLogy();
ppac1_Ty_Delay->Draw();
c1->Draw();

## ppac1_dty21

In [ ]:
spileup.Form("abs(p1_Ty_Delay-%f)<3*%f",ppac_ypeak[0],ppac_ysigma[0]);
TCut cpileup = spileup.Data();
tree->Draw("p1_dty21>>ppac1_dty21(4000,-200,200)", p1_cy1 && p1_cy2 && cpileup, "goff");
ppac1_dty21= (TH1D*) gROOT->FindObject("ppac1_dty21");

In [ ]:
nfound = s_ppac1_dty21->Search(ppac1_dty21,3,"",0.001);
dt = s_ppac1_dty21->GetPositionX();
// dx_count = s_dty21->GetPositionY();
sort(dt,dt+nfound);
select(3.25,3.75);
for (int i=0; i<8; i++)
{
//     cout << dt[i+v1[0]] << endl;
    fy_dt[0][i] = new TF1(Form("f0%d", i), "gaus");
    ppac1_dty21->Fit(fx_dt[0][i],"SQ+","",dt[i+v1[0]+1-10]-0.6,dt[i+v1[0]+1-10]+0.6);//fp1
//      ppac1_dty21->Fit(fx_dt[0][i],"SQ+","",dt[i+v1[1]+1-10]-0.6,dt[i+v1[1]+1-10]+0.6);//fp1
//     fx[count] = dty21->GetFunction("gaus");
    ppac_dty[0][i] = fx_dt[0][i]->GetParameter(1);
    // xsigma[i] = fx[i]->GetParameter(2);
    // cout << xpeak[i] <<" + "<< xsigma << endl;
}

gPad->SetLogy(0);
ppac1_dty21->Draw();
c1->Draw();
v1.clear();

In [ ]:
gr = new TGraph(8,ppac_dty[0],position);
gr->Draw("AQ*");
gr->Fit("pol1","Q");
fit_y[0] = gr->GetFunction("pol1");

ppac_corey[0][0] = fit_y[0]->GetParameter(0);
ppac_corey[0][1] = fit_y[0]->GetParameter(1);
ppac_corey[0][2] = fit_y[0]->GetChisquare()/fit_y[0]->GetNDF();
cout << Form("p0 = %.6f, p1 = %.6f, chi2/ndf = %.6f", ppac_corey[0][0], ppac_corey[0][1], ppac_corey[0][2]) << endl;

c1->Draw();

## ppac2_Tx_Delay

In [ ]:
//ppac2_Tx_Delay
tree->Draw("p2_Tx_Delay>>ppac2_Tx_Delay(6000,-200,400)", p2_cx1 && p2_cx2, "goff");
ppac2_Tx_Delay = (TH1D*) gROOT->FindObject("ppac2_Tx_Delay");

In [ ]:
nfound = s_ppac2_Tx_Delay->Search(ppac2_Tx_Delay,0.7,"",0.9);
dt = s_ppac2_Tx_Delay->GetPositionX();
ppac2_Tx_Delay->Fit("gaus","Q","", dt[0]-1.2, dt[0]+1.2);
fx_delay[1] = ppac2_Tx_Delay->GetFunction("gaus");
ppac_xpeak[1] = fx_delay[1]->GetParameter(1);
ppac_xsigma[1] = fx_delay[1]->GetParameter(2);

cout << ppac_xpeak[1] << " + " << ppac_xsigma[1] << endl;
gPad->SetLogy();
ppac2_Tx_Delay->Draw();
c1->Draw();

## ppac2_dtx12

In [ ]:
spileup.Form("abs(p2_Tx_Delay-%f)<3*%f",ppac_xpeak[1],ppac_xsigma[1]);
TCut cpileup = spileup.Data();
tree->Draw("p2_dtx12>>ppac2_dtx12(4000,-200,200)", p2_cx1 && p2_cx2 && cpileup, "goff");
ppac2_dtx12= (TH1D*) gROOT->FindObject("ppac2_dtx12");

In [ ]:
nfound = s_ppac2_dtx12->Search(ppac2_dtx12,2,"",0.1);
dt = s_ppac2_dtx12->GetPositionX();
// dx_count = s_dtx12->GetPositionY();
sort(dt,dt+nfound);
select(3.25,3.75);
for (int i=0; i<8; i++)
{
//     cout << dt[i+v1[0]] << endl;
    fx_dt[1][i] = new TF1(Form("f0%d", i), "gaus");
    ppac2_dtx12->Fit(fx_dt[1][i],"SQ+","",dt[i+v1[0]+1]-0.6,dt[i+v1[0]+1]+0.6);//fp2
//     fx[count] = dtx12->GetFunction("gaus");
    ppac_dtx[1][i] = fx_dt[1][i]->GetParameter(1);
    // xsigma[i] = fx[i]->GetParameter(2);
    // cout << xpeak[i] <<" + "<< xsigma << endl;
}

gPad->SetLogy(0);
ppac2_dtx12->Draw();
c1->Draw();
v1.clear();


In [ ]:
gr = new TGraph(8,ppac_dtx[1],position);
gr->Draw("AQ*");
gr->Fit("pol1","Q");
fit_x[1] = gr->GetFunction("pol1");

ppac_corex[1][0] = fit_x[1]->GetParameter(0);
ppac_corex[1][1] = fit_x[1]->GetParameter(1);
ppac_corex[1][2] = fit_x[1]->GetChisquare()/fit_x[1]->GetNDF();
cout << Form("p0 = %.6f, p1 = %.6f, chi2/ndf = %.6f", ppac_corex[1][0], ppac_corex[1][1], ppac_corex[1][2]) << endl;

c1->Draw();

## ppac2_Ty_Delay


In [ ]:
tree->Draw("p2_Ty_Delay>>ppac2_Ty_Delay(6000,-200,400)", p2_cy1 && p2_cy2, "goff");
ppac2_Ty_Delay = (TH1D*) gROOT->FindObject("ppac2_Ty_Delay");

In [ ]:
nfound = s_ppac2_Ty_Delay->Search(ppac2_Ty_Delay,0.7,"",0.9);
dt = s_ppac2_Ty_Delay->GetPositionX();
ppac2_Ty_Delay->Fit("gaus","Q","", dt[0]-1.2, dt[0]+1.2);
fy_delay[1] = ppac2_Ty_Delay->GetFunction("gaus");
ppac_ypeak[1] = fy_delay[1]->GetParameter(1);
ppac_ysigma[1] = fy_delay[1]->GetParameter(2);

cout << ppac_ypeak[1] << " + " << ppac_ysigma[1] << endl;
gPad->SetLogy();
ppac2_Ty_Delay->Draw();
c1->Draw();

## ppac2_dty21

In [ ]:
spileup.Form("abs(p2_Ty_Delay-%f)<3*%f",ppac_ypeak[1],ppac_ysigma[1]);
TCut cpileup = spileup.Data();
tree->Draw("p2_dty21>>ppac2_dty21(4000,-200,200)", p2_cy1 && p2_cy2 && cpileup, "goff");
ppac2_dty21= (TH1D*) gROOT->FindObject("ppac2_dty21");

In [ ]:
nfound = s_ppac2_dty21->Search(ppac2_dty21,2,"",0.005);
dt = s_ppac2_dty21->GetPositionX();
// dx_count = s_dty21->GetPositionY();
sort(dt,dt+nfound);
select(3.25,3.75);
for (int i=0; i<8; i++)
{
//     cout << dt[i+v1[0]] << endl;
    fy_dt[1][i] = new TF1(Form("f0%d", i), "gaus");
//     ppac2_dty21->Fit(fx_dt[1][i],"SQ+","",dt[i+v1[0]+1]-0.6,dt[i+v1[0]+1]+0.6);//fp1
    ppac2_dty21->Fit(fx_dt[1][i],"SQ+","",dt[i+v1[0]+1-10]-0.6,dt[i+v1[0]+1-10]+0.6);//fp1
//     ppac2_dty21->Fit(fx_dt[1][i],"SQ+","",dt[i+v1[1]+1-10]-0.6,dt[i+v1[1]+1-10]+0.6);//fp1
//     fx[count] = dty21->GetFunction("gaus");
    ppac_dty[1][i] = fx_dt[1][i]->GetParameter(1);
    // xsigma[i] = fx[i]->GetParameter(2);
    // cout << xpeak[i] <<" + "<< xsigma << endl;
}

gPad->SetLogy(0);
ppac2_dty21->Draw();
c1->Draw();
v1.clear();

In [ ]:
gr = new TGraph(8,ppac_dty[1],position);
gr->Draw("AQ*");
gr->Fit("pol1","Q");
fit_y[1] = gr->GetFunction("pol1");

ppac_corey[1][0] = fit_y[1]->GetParameter(0);
ppac_corey[1][1] = fit_y[1]->GetParameter(1);
ppac_corey[1][2] = fit_y[1]->GetChisquare()/fit_y[1]->GetNDF();
cout << Form("p0 = %.6f, p1 = %.6f, chi2/ndf = %.6f", ppac_corey[1][0], ppac_corey[1][1], ppac_corey[1][2]) << endl;

c1->Draw();

## ppac3_Tx_Delay

In [ ]:
//ppac3_Tx_Delay
tree->Draw("p3_Tx_Delay>>ppac3_Tx_Delay(6000,-200,400)", p3_cx1 && p3_cx2, "goff");
ppac3_Tx_Delay = (TH1D*) gROOT->FindObject("ppac3_Tx_Delay");

In [ ]:
nfound = s_ppac3_Tx_Delay->Search(ppac3_Tx_Delay,0.7,"",0.9);
dt = s_ppac3_Tx_Delay->GetPositionX();
ppac3_Tx_Delay->Fit("gaus","Q","", dt[0]-1.2, dt[0]+1.2);
fx_delay[2] = ppac3_Tx_Delay->GetFunction("gaus");
ppac_xpeak[2] = fx_delay[2]->GetParameter(1);
ppac_xsigma[2] = fx_delay[2]->GetParameter(2);

cout << ppac_xpeak[2] << " + " << ppac_xsigma[2] << endl;
gPad->SetLogy();
ppac3_Tx_Delay->Draw();
c1->Draw();

## ppac3_dtx12


In [ ]:
spileup.Form("abs(p3_Tx_Delay-%f)<3*%f",ppac_xpeak[2],ppac_xsigma[2]);
TCut cpileup = spileup.Data();
tree->Draw("p3_dtx12>>ppac3_dtx12(4000,-200,200)", p3_cx1 && p3_cx2 && cpileup, "goff");
ppac3_dtx12= (TH1D*) gROOT->FindObject("ppac3_dtx12");

In [ ]:
nfound = s_ppac3_dtx12->Search(ppac3_dtx12,2,"",0.1);
dt = s_ppac3_dtx12->GetPositionX();
// dx_count = s_dtx12->GetPositionY();
sort(dt,dt+nfound);
select(3.25,3.75);
for (int i=0; i<8; i++)
{
//     cout << dt[i+v1[0]] << endl;
    fx_dt[2][i] = new TF1(Form("f0%d", i), "gaus");
    ppac3_dtx12->Fit(fx_dt[2][i],"SQ+","",dt[i+v1[0]+1]-0.6,dt[i+v1[0]+1]+0.6);//fp2
//     fx[count] = dtx12->GetFunction("gaus");
    ppac_dtx[2][i] = fx_dt[2][i]->GetParameter(1);
    // xsigma[i] = fx[i]->GetParameter(2);
    // cout << xpeak[i] <<" + "<< xsigma << endl;
}

gPad->SetLogy(0);
ppac3_dtx12->Draw();
c1->Draw();
v1.clear();

In [ ]:
gr = new TGraph(8,ppac_dtx[2],position);
gr->Draw("AQ*");
gr->Fit("pol1","Q");
fit_x[2] = gr->GetFunction("pol1");

ppac_corex[2][0] = fit_x[2]->GetParameter(0);
ppac_corex[2][1] = fit_x[2]->GetParameter(1);
ppac_corex[2][2] = fit_x[2]->GetChisquare()/fit_x[2]->GetNDF();
cout << Form("p0 = %.6f, p1 = %.6f, chi2/ndf = %.6f", ppac_corex[2][0], ppac_corex[2][1], ppac_corex[2][2]) << endl;

c1->Draw();

## ppac3_Ty_Delay

In [ ]:
tree->Draw("p3_Ty_Delay>>ppac3_Ty_Delay(6000,-200,400)", p3_cy1 && p3_cy2, "goff");
ppac3_Ty_Delay = (TH1D*) gROOT->FindObject("ppac3_Ty_Delay");

In [ ]:
nfound = s_ppac3_Ty_Delay->Search(ppac3_Ty_Delay,0.7,"",0.9);
dt = s_ppac3_Ty_Delay->GetPositionX();
ppac3_Ty_Delay->Fit("gaus","Q","", dt[0]-1.2, dt[0]+1.2);
fy_delay[2] = ppac3_Ty_Delay->GetFunction("gaus");
ppac_ypeak[2] = fy_delay[2]->GetParameter(1);
ppac_ysigma[2] = fy_delay[2]->GetParameter(2);

cout << ppac_ypeak[2] << " + " << ppac_ysigma[2] << endl;
gPad->SetLogy();
ppac3_Ty_Delay->Draw();
c1->Draw();

## ppac3_dty21

In [ ]:
spileup.Form("abs(p3_Ty_Delay-%f)<3*%f",ppac_ypeak[2],ppac_ysigma[2]);
TCut cpileup = spileup.Data();
tree->Draw("p3_dty21>>ppac3_dty21(4000,-200,200)", p3_cy1 && p3_cy2 && cpileup, "goff");
ppac3_dty21= (TH1D*) gROOT->FindObject("ppac3_dty21");

In [ ]:
nfound = s_ppac3_dty21->Search(ppac3_dty21,2,"",0.005);
dt = s_ppac3_dty21->GetPositionX();
// dx_count = s_dty21->GetPositionY();
sort(dt,dt+nfound);
select(3.25,3.75);
for (int i=0; i<8; i++)
{
//     cout << dt[i+v1[0]] << endl;
    fy_dt[2][i] = new TF1(Form("f0%d", i), "gaus");
    ppac3_dty21->Fit(fx_dt[2][i],"SQ+","",dt[i+v1[0]+1]-0.6,dt[i+v1[0]+1]+0.6);//fp1
//     fx[count] = dty21->GetFunction("gaus");
    ppac_dty[2][i] = fx_dt[2][i]->GetParameter(1);
    // xsigma[i] = fx[i]->GetParameter(2);
    // cout << xpeak[i] <<" + "<< xsigma << endl;
}

gPad->SetLogy(0);
ppac3_dty21->Draw();
c1->Draw();
v1.clear();

In [ ]:
gr = new TGraph(8,ppac_dty[2],position);
gr->Draw("AQ*");
gr->Fit("pol1","Q");
fit_y[2] = gr->GetFunction("pol1");

ppac_corey[2][0] = fit_y[2]->GetParameter(0);
ppac_corey[2][1] = fit_y[2]->GetParameter(1);
ppac_corey[2][2] = fit_y[2]->GetChisquare()/fit_y[2]->GetNDF();
cout << Form("p0 = %.6f, p1 = %.6f, chi2/ndf = %.6f", ppac_corey[2][0], ppac_corey[2][1], ppac_corey[2][2]) << endl;

c1->Draw();

In [ ]:
for (int i =0; i<3; i++)
{
    cout << "ppac" << i+1 <<endl; 
    cout << Form("x_peak = %.6f, x_sigma = %.6f", ppac_xpeak[i], ppac_xsigma[i]) << endl;
    cout << Form("y_peak = %.6f, y_sigma = %.6f", ppac_ypeak[i], ppac_ysigma[i]) << endl;
    cout << Form("p0_x   = %.6f, p1_x    = %.6f, chi2/ndf_x = %.6f", ppac_corex[i][0], ppac_corex[i][1], ppac_corex[i][2]) << endl;
    cout << Form("p0_y   = %.6f, p1_y    = %.6f, chi2/ndf_y = %.6f", ppac_corey[i][0], ppac_corey[i][1], ppac_corey[i][2]) << endl;
}

In [ ]:
 ppac1_Tx_Delay->Write();
 ppac1_Ty_Delay->Write();
 ppac1_dtx12->Write();
 ppac1_dty21->Write();

 ppac2_Tx_Delay->Write();
 ppac2_Ty_Delay->Write();
 ppac2_dtx12->Write();
 ppac2_dty21->Write();

 ppac3_Tx_Delay->Write();
 ppac3_Ty_Delay->Write();
 ppac3_dtx12->Write();
 ppac3_dty21->Write();

In [ ]:
TString outfile_txt;
outfile_txt.Form("%sppac-correlation-%04d.txt", OutputPath.Data(), run_number);
cout << outfile_txt << endl;


In [ ]:
std::ofstream outFile(outfile_txt);
outFile << Form("%7s%10s%10s%10s%10s%10s%10s%10s%10s\n","ppac_id","x_peak","x_sigma","y_peak","y_sigma","p0_x","p1_x","p0_y","p1_y");
for (int i=0; i<3; i++)
{
    outFile << Form("%5d%13.4f%10.4f%10.4f%10.4f%10.4f%10.4f%10.4f%10.4f\n",i+1,ppac_xpeak[i],ppac_xsigma[i],ppac_ypeak[i],ppac_ysigma[i],ppac_corex[i][0], ppac_corex[i][1],ppac_corey[i][0], ppac_corey[i][1]);
}
outFile.close();

## 对于各项效率指标的计算

In [ ]:
// double total_nb = tree->GetEntries();
// double nb_p1_ca = tree->GetEntries(p1_ca);
// double nb_p1_x  = tree->GetEntries(p1_cx1 && p1_cx2);
// double nb_p1_y  = tree->GetEntries(p1_cy1 && p1_cy2);
// cout << "ppac1" <<endl; 
// cout << "" << endl;
// cout << "阳极效率：" <<Form("%.4f %%", nb_p1_ca/total_nb *100.) << endl;
// cout << "x方向阴极效率（阳极和阴极同时有效/阳极有效）：" <<Form("%.4f %%", nb_p1_x/nb_p1_ca *100.) << endl;
// cout << "y方向阴极效率（阳极和阴极同时有效/阳极有效）：" <<Form("%.4f %%", nb_p1_y/nb_p1_ca *100.) << endl;
// cout << "---------------------------------------------" << endl;


In [ ]:
// double total_nb = tree->GetEntries();
// double nb_p2_ca = tree->GetEntries(p2_ca);
// double nb_p2_x  = tree->GetEntries(p2_cx1 && p2_cx2);
// double nb_p2_y  = tree->GetEntries(p2_cy1 && p2_cy2);
// cout << "ppac2" <<endl; 
// cout << "" << endl;
// cout << "阳极效率：" <<Form("%.4f %%", nb_p2_ca/total_nb *100.) << endl;
// cout << "x方向阴极效率（阳极和阴极同时有效/阳极有效）：" <<Form("%.4f %%", nb_p2_x/nb_p2_ca *100.) << endl;
// cout << "y方向阴极效率（阳极和阴极同时有效/阳极有效）：" <<Form("%.4f %%", nb_p2_y/nb_p2_ca *100.) << endl;
// cout << "---------------------------------------------" << endl;


In [ ]:
// double total_nb = tree->GetEntries();
// double nb_p3_ca = tree->GetEntries(p3_ca);
// double nb_p3_x  = tree->GetEntries(p3_cx1 && p3_cx2);
// double nb_p3_y  = tree->GetEntries(p3_cy1 && p3_cy2);
// cout << "ppac3" <<endl; 
// cout << "" << endl;
// cout << "阳极效率：" <<Form("%.4f %%", nb_p3_ca/total_nb *100.) << endl;
// cout << "x方向阴极效率（阳极和阴极同时有效/阳极有效）：" <<Form("%.4f %%", nb_p3_x/nb_p3_ca *100.) << endl;
// cout << "y方向阴极效率（阳极和阴极同时有效/阳极有效）：" <<Form("%.4f %%", nb_p3_y/nb_p3_ca *100.) << endl;
// cout << "---------------------------------------------" << endl;


In [ ]:
opf->Close();